## Problem Statement

### Business Context

Modern organizations operate in an environment where internal policies have grown increasingly complex, covering areas such as leaves, reimbursements, travel, hybrid work norms, performance management, compliance, and more. These policies are commonly documented in static, lengthy HR handbooks or PDFs, which employees often find difficult to understand, search, or apply in their daily work scenarios.

As a result:
- HR teams are burdened with repetitive queries about standard policies, diverting their focus from strategic initiatives.
- Employees struggle with confusion or non-compliance, as the dense and static format of policy documents obscures the necessary guidance.
- Both sides suffer from reduced productivity, HR spends excessive time addressing routine questions, while employees experience delays in obtaining the information they need to perform efficiently.

Addressing these challenges requires modern HR systems that centralize policy information, simplify access, and deliver clear, actionable insights. By leveraging technology to streamline policy communication and automate routine queries, organizations can enhance clarity, boost compliance, and ultimately improve overall operational efficiency.

###  Objective

The goal is to develop a prototype that demonstrates how Natural Language Processing (NLP), powered by Retrieval-Augmented Generation (RAG), can help employees efficiently query company HR policies and receive accurate, context-aware, and easily understandable answers.

Specifically, the system aims to:

- Answer employee questions by retrieving relevant content from official HR handbooks and policy documents.
- Handle ambiguous queries and follow-up questions by clarifying intent and distinguishing between similar policy categories (e.g., sick leave versus casual leave).
- Personalize responses based on role, location, or department, acknowledging that policies may differ (e.g., field staff versus HQ).
- Increase trust and compliance by citing sources (document name, section, and clause) for each response.


**Common Questions to Answer**

1. What are the effects on the benefits I receive if my probation is extended?

2. There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

3. What should I do if I notice suspected harassment with my female colleague?

### Data Description

The dataset consists of an employee handbook from Flykite Airlines in PDF format.


## Installing and Importing Necessary Libraries and Dependencies

In [6]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

Import core libraries and dependencies

In [7]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

Mounting the drive

In [8]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Load the JSON file and extract values
file_name = '/content/drive/MyDrive/Colab Notebooks/config.json'                                                     # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                      # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

## Question Answering using LLM

#### Downloading and Loading the model

In [32]:
# Define a function to get the response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
  # Create a chat completion using the OpenAI Client
  completion = client.chat.completions.create(
      model = "gpt-4o-mini",
      messages = [
           {"role": "user", "content": user_prompt}
      ],
      max_tokens = max_tokens,
      temperature = temperature,
      top_p = top_p
  )
  return completion.choices[0].message.content

### Question 1: What are the effects on the benefits I receive if my probation is extended?

In [33]:
question_1= "What are the effects on the benefits I receive if my probation is extended?"
base_prompt_response_1 = response(question_1)
print(base_prompt_response_1)

The effects on the benefits you receive due to an extension of your probation can vary depending on several factors, including the type of benefits you receive and the policies of your specific employer or jurisdiction. Here are some potential impacts:

1. **Employment Benefits**: If you are on probationary status, you may not be eligible for certain employment benefits, such as health insurance, retirement contributions, or paid time off. An extension of your probation could delay your access to these benefits.

2. **Job Security**: Extended probation may affect your job security. If your employer has concerns about your performance, they may choose to terminate your employment rather than confirm your position.

3. **Performance Reviews**: An extended probation may mean additional performance reviews or evaluations. This could impact your potential for promotions or salary increases.

4. **Unemployment Benefits**: If your probation is extended and you are subsequently terminated, you

### Question 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [34]:
question_2 = "There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?"
base_prompt_response_2 = response(question_2)
base_prompt_response_2

"I'm very sorry to hear about your loss. When informing your office about the situation, it's important to communicate clearly and professionally. Here’s a suggested approach for how you might draft your message:\n\n---\n\n**Subject:** Request for Leave Due to Family Emergency\n\nDear [Supervisor’s Name/HR Department],\n\nI hope this message finds you well. I am writing to inform you that I experienced a bereavement in my family last night. Due to this unforeseen circumstance, I will need to attend the last rites and support my family during this difficult time.\n\nI kindly request [number of days] days of leave, starting from [start date] to [end date]. I will ensure that all my responsibilities are managed and will provide any necessary handover details to ensure a smooth workflow in my absence.\n\nThank you for your understanding during this challenging time. I will keep you updated and will do my best to stay in touch as needed.\n\nSincerely,  \n[Your Name]  \n[Your Position]  \n[Y

### Question 3: What should I do if I notice suspected harassment with my female colleague?

In [35]:
question_3 = "What should I do if I notice suspected harassment with my female colleague?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

"If you notice suspected harassment involving a female colleague, it's important to approach the situation thoughtfully and responsibly. Here are steps you can take:\n\n1. **Assess the Situation**: Ensure that what you’re observing is indeed harassment. Look for repeated unwanted behavior, comments, or actions that create a hostile environment.\n\n2. **Document Everything**: Keep a record of the incidents you observe, including dates, times, locations, what was said or done, and any witnesses. This documentation can be crucial if formal action is needed later.\n\n3. **Support Your Colleague**: If you feel comfortable, approach your colleague privately and express your concern. Let her know that you’ve noticed what’s been happening and that you’re there to support her. Respect her wishes if she prefers not to discuss it or take action.\n\n4. **Encourage Reporting**: If she feels comfortable, encourage her to report the harassment to a supervisor, HR, or the appropriate department within

## Data Preparation for RAG

### Loading the Data

In [10]:
# Set the path to the PDF file
manual_pdf_path = "/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()

In [11]:
# Define a function to get the response
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
  # Create a chat completion using the OpenAI Client
  completion = client.chat.completions.create(
      model = "gpt-4o-mini",
      messages = [
           {"role": "user", "content": user_prompt},
           {"role": "system", "content": system_prompt}
      ],
      max_tokens = max_tokens,
      temperature = temperature,
      top_p = top_p
  )
  return completion.choices[0].message.content

In [38]:
# Define the system prompt for the model
qna_system_message = """ You as a modern Human Resources (HR) AI Assistant centralize the policy information, simplify access, and deliver clear, actionable insights.
You should help employees efficiently query company HR policies and receive accurate, context-aware, and easily understandable answers.
The response you give is entirely based on the context provided.  User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of FlykiteAirlines:HumanResourcesPolicyHandbook relevant to the query, along with their source details.When crafting your response:
This context will begin with the token: ###Context.

1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."


Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.



Here is an example of how to structure your response:


Answer:
[answer based on context]


Source:
[Source details with page or section]
"""

In [41]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from Flykite Airlines HRP PDF and their sources that are relevant to the question mentioned below:
{context}

###Question
{question}
"""

### Data Overview

In [12]:
for i, doc in enumerate(manual):
    print(f"--- Page {i} | {len(doc.page_content)} chars ---")
    print(doc.page_content[:600])
    print()

--- Page 0 | 1406 chars ---
Flykite Airlines: Human Resources Policy 
Handbook 
Introduction 
Flykite Airlines is dedicated to cultivating an organizational culture that synergizes 
operational excellence with a supportive, equitable, and legally compliant workplace 
environment across all departments and employee levels. This document establishes 
an exhaustive framework comprising all human resource policies currently in effect. All 
provisions are subject to amendment, interpretation, or rescindment at the sole 
discretion of the Human Resources and Legal departments. In the event of ambiguities 
or conflicting interp

--- Page 1 | 1336 chars ---
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of 

In [15]:
#Initialize the text_splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name = 'cl100k_base',
    chunk_size = 256,
    chunk_overlap = 20
    )


### Split the pdf into chunks for further processing

In [17]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)

### Generate Vector Embeddings for Text Chunks Using OpenAI


In [19]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))

Dimension of the embedding vector  1536


### Vector Database

### Setup vector Store directory

In [20]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'FlykiteAirlinesRules_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)


### Create Vector store from Documents

In [21]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

### Load Vector Store

In [23]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

In [25]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7d741098c690>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7d741098cf50>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX18L5UirQsZlYI1XkQ3VoJLljWg9bhvoowctTiH1DXkXR+ycP4fnxsg/', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [26]:
# Search for top 3 most relevant chunks for the query
vectorstore.similarity_search(
    "What are the effects on the benefits I receive if my probation is extended?",
    k=3
)

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'title': 'Flykite Airlines: HRP', 'page': 1, 'keywords': '', 'author': '', 'modDate': 'D:20260824203932Z', 'format': 'PDF 1.7', 'source': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf', 'moddate': '2026-08-24T20:39:32+00:00', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'total_pages': 14, 'subject': '', 'file_path': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf', 'creationDate': 'D:20260824203932Z', 'trapped': '', 'creationdate': '2026-08-24T20:39:32+00:00'}, page_content='●\u200b Extensions are granted only if:\u200b\n a. The employee has achieved at least 60% of their probationary objectives.\u200b\n b. A written PIP with measurable targets is issued within 5 working days of the \noriginal probation end date.\u200b\n c. The Department Head and HR Manager both sign off on the extension \nrequest.\u200b\n

### Retriever

### Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [27]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 3 most relevant documents
)

### Response Function

In [29]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    print("relevant chunks ", relevant_document_chunks)
    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: What are the effects on the benefits I receive if my probation is extended?

In [44]:
response_with_rag_1 = generate_rag_response(question_1, k=3)

response_with_rag_1

relevant chunks  [Document(metadata={'total_pages': 14, 'moddate': '2026-08-24T20:39:32+00:00', 'format': 'PDF 1.7', 'page': 1, 'creationdate': '2026-08-24T20:39:32+00:00', 'keywords': '', 'modDate': 'D:20260824203932Z', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'trapped': '', 'subject': '', 'creationDate': 'D:20260824203932Z', 'author': '', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'file_path': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf', 'title': 'Flykite Airlines: HRP', 'source': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf'}, page_content='●\u200b Extensions are granted only if:\u200b\n a. The employee has achieved at least 60% of their probationary objectives.\u200b\n b. A written PIP with measurable targets is issued within 5 working days of the \noriginal probation end date.\u200b\n c. The Department Head and HR Manager both sign off on the extension \

'Answer:\nWhile on probation, including any extension, employees are not eligible for:\n- Annual leave encashment\n- Internal role transfers\n- Performance bonuses\n\nAdditionally, seniority accrual starts only after successful completion of probation.\n\nSource:\nFlykite Airlines HRP PDF, Section 3: Impact on Benefits, Seniority, and Contract.'

Answer:
While on probation, including any extension, employees are not eligible for:
- Annual leave encashment
- Internal role transfers
- Performance bonuses

Additionally, seniority accrual starts only after successful completion of probation.

Source:
Flykite Airlines HRP PDF, Section 3: Impact on Benefits, Seniority, and Contract.

### Question 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [45]:
response_with_rag_2 = generate_rag_response(question_2, k=3)

response_with_rag_2

relevant chunks  [Document(metadata={'format': 'PDF 1.7', 'modDate': 'D:20260824203932Z', 'total_pages': 14, 'creationdate': '2026-08-24T20:39:32+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'title': 'Flykite Airlines: HRP', 'subject': '', 'file_path': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf', 'author': '', 'moddate': '2026-08-24T20:39:32+00:00', 'creationDate': 'D:20260824203932Z', 'source': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf', 'trapped': '', 'keywords': '', 'page': 7}, page_content='leave, except for emergencies.\u200b\n \n●\u200b For sudden emergencies:\u200b\n \n○\u200b Verbal notification within 6 hours of incident.\u200b\n \nnshk0816@gmail.com\nA06KZ4L3O9\nThis file is meant for personal use by nshk0816@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal acti

'Answer:\nYou should provide a verbal notification to the office within 6 hours of the incident. You are eligible for emergency family care leave, which allows for up to 3 working days per calendar year.\n\nSource:\nFlykite Airlines HRP, page not specified.'

Answer:
You should provide a verbal notification to the office within 6 hours of the incident. You are eligible for emergency family care leave, which allows for up to 3 working days per calendar year.

Source:
Flykite Airlines HRP, page not specified.

### Question 3: What should I do if I notice suspected harassment with my female colleague?

In [46]:
response_with_rag_3 = generate_rag_response(question_3, k=3)

response_with_rag_3

relevant chunks  [Document(metadata={'total_pages': 14, 'modDate': 'D:20260824203932Z', 'subject': '', 'creationDate': 'D:20260824203932Z', 'creationdate': '2026-08-24T20:39:32+00:00', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'moddate': '2026-08-24T20:39:32+00:00', 'page': 4, 'format': 'PDF 1.7', 'title': 'Flykite Airlines: HRP', 'trapped': '', 'file_path': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'author': '', 'source': '/content/drive/MyDrive/Colab Notebooks/RAG_Flykite Airlines HRP/Flykite Airlines_ HRP.pdf', 'keywords': ''}, page_content='●\u200b Accepted reporting channels:\u200b\n \n1.\u200b Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 \nPM IST).\u200b\n \n2.\u200b Secure Email Portal: hr.ethics@flykiteair.com.\u200b\n \n3.\u200b Anonymous drop-boxes in crew lounges and staff cafeterias.\u200b\n \n●\u200b Reports must include d

'Answer:\nIf you notice suspected harassment with your female colleague, you must report the incident within 15 calendar days of occurrence or discovery. You can use the following reporting channels:\n\n1. Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 PM IST).\n2. Secure Email Portal: hr.ethics@flykiteair.com.\n3. Anonymous drop-boxes in crew lounges and staff cafeterias.\n\nMake sure your report includes the date, location, persons involved, and a brief incident description.\n\nSource:\nFlykite Airlines HRP PDF, relevant sections on reporting and protection against retaliation.'

Answer:
If you notice suspected harassment with your female colleague, you must report the incident within 15 calendar days of occurrence or discovery. You can use the following reporting channels:

1. Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 PM IST).
2. Secure Email Portal: hr.ethics@flykiteair.com.
3. Anonymous drop-boxes in crew lounges and staff cafeterias.

Make sure your report includes the date, location, persons involved, and a brief incident description.

Source:
Flykite Airlines HRP PDF, relevant sections on reporting and protection against retaliation.